# 06 — Hybrid Retrieval: BM25 + MedCPT via RRF (M5)

Runs `scripts/run_hybrid.py`: fuses the BM25 (M2) and MedCPT (M4) runs with Reciprocal Rank Fusion, `RRF(d) = Σ 1/(k + rank_i(d))`, k=60 default (`configs/hybrid.yaml`). Operates on **rank positions**, not raw scores — BM25 and MedCPT scores are not on comparable scales (see `docs/architecture.md`).

**Prerequisite:** requires `results/runs/{bm25,medcpt}.trec` to already exist (notebooks 03 and 05).

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
import time
start = time.perf_counter()
result = subprocess.run([sys.executable, 'scripts/run_hybrid.py'], cwd=os.getcwd())
print(f'\nexit code: {result.returncode}, elapsed: {time.perf_counter()-start:.1f}s')
assert result.returncode == 0, 'Script failed -- see output above.'

## Real results

In [ ]:
import json
payload = json.load(open('results/metrics/hybrid_rrf.json'))
print(json.dumps(payload, indent=2)[:3000])

## Hand-computable RRF sanity check

`src/biomedical_ir/fusion.py` is unit-tested against hand-created rankings (Section 10 of the project spec) — reproduced here inline:

In [ ]:
from biomedical_ir.fusion import reciprocal_rank_fusion

# ranking1 = [A, B, C], ranking2 = [B, A, D]. With k=1:
# RRF(A) = 1/(1+1) + 1/(1+2) = 0.8333...  RRF(B) = 1/(1+2) + 1/(1+1) = 0.8333... (tie)
# RRF(C) = 1/(1+3) + 0 = 0.25              RRF(D) = 0 + 1/(1+3) = 0.25 (tie)
result = reciprocal_rank_fusion([['A', 'B', 'C'], ['B', 'A', 'D']], k=1)
print(result)
assert [d for d, _ in result] == ['A', 'B', 'C', 'D']  # ties broken by ascending doc_id

**M7 finding:** MedCPT vs. Hybrid RRF found MedCPT significantly *beats* Hybrid RRF on Recall@100 (p=0.040) — the opposite of H3's predicted direction. No significant difference on P@10/MAP/MRR@10/nDCG@10.